# VHAGAR: physics-informed next-day fire spread (Colab)

Calibrated **physics prior** (fast-marching front) vs a **U-Net corrector** that also
sees the layers the prior ignores, scored on the incremental new-burn region with
discrimination + calibration proper scores, under **leave-fire-out** CV. Frontier task
(WildfireSpreadTS), done the VHAGAR way.

Run *Runtime -> GPU*. Two modes: a synthetic demo (no data), or real WildfireSpreadTS
GeoTIFF pairs from Drive.


## 0. Install + repo (Colab has torch; don't touch numpy)


In [ ]:
REPO='https://github.com/Ibekwemmanuel7/VHAGAR.git'; REPO_DIR='/content/VHAGAR'
import os
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO $REPO_DIR || echo 'clone failed (private? use a token)'
%pip install -q rasterio            # only for real GeoTIFFs; Colab has the rest
import sys; sys.path.insert(0, f'{REPO_DIR}/src')
import torch; print('torch', torch.__version__, '| GPU', torch.cuda.is_available())


## 1a. Synthetic demo (physics prior + corrector, no data)
Confirms the pipeline and shows whether the corrector beats the calibrated prior.


In [ ]:
import numpy as np
from vhagar.datasets.wildfirespread import synthetic_wfs_fire
from vhagar.eval.wildfirespread import evaluate_nextday
samples={}
for i in range(24):
    s,_=synthetic_wfs_fire(np.random.default_rng(500+i), fire_id=f'fire{i:02d}')
    samples[s.fire_id]=s
rep=evaluate_nextday(samples, k=5, horizon=18.0, calibrate=True, with_corrector=True, epochs=25)
for m,v in rep['summary'].items():
    print(f"{m:20s} AP {v['ap_mean']:.3f}  F1 {v['f1_mean']:.3f}  IoU {v['iou_mean']:.3f}  "
          f"Brier {v['brier_mean']:.4f}  ECE {v['ece_mean']:.3f}  ({v['fires']} fires)")
for n in rep['notes']: print('note:', n)


## 1b. Real WildfireSpreadTS
Download WildfireSpreadTS (Gerard et al. 2023) to Drive: each fire is a folder of daily
multi-band GeoTIFFs. Set `WFS_ROOT`, and map our channel names to the dataset's band
indices (see its data card). Then it loads consecutive day pairs per fire and runs the
same leave-fire-out evaluation.


In [ ]:
import glob, numpy as np
from vhagar.datasets.wildfirespread import load_wfs_geotiff_pair, CHANNELS
from vhagar.eval.wildfirespread import evaluate_nextday

WFS_ROOT='/content/drive/MyDrive/WildfireSpreadTS'   # <- fires/*/<date>.tif
# ADAPT to the dataset's band order (indices into each daily stack):
CHANNEL_MAP={'fuel':16, 'wind':4, 'slope':13, 'barrier':21}   # example indices
FIRE_CHANNEL=22                                                # VIIRS active-fire band

from google.colab import drive; drive.mount('/content/drive')
samples={}
for fire_dir in sorted(glob.glob(f'{WFS_ROOT}/*')):
    days=sorted(glob.glob(f'{fire_dir}/*.tif'))
    fid=os.path.basename(fire_dir)
    for a,b in zip(days, days[1:]):        # consecutive day pairs
        try:
            s=load_wfs_geotiff_pair(a,b, channel_map=CHANNEL_MAP, fire_channel=FIRE_CHANNEL,
                                    fire_id=f'{fid}:{os.path.basename(a)[:-4]}')
            if s.is_usable: samples[s.fire_id]=s
        except Exception as e:
            pass
print(len(samples),'usable day-pairs')
rep=evaluate_nextday(samples, k=5, horizon=18.0, calibrate=True, with_corrector=True, epochs=25)
for m,v in rep['summary'].items():
    print(f"{m:20s} AP {v['ap_mean']:.3f}  F1 {v['f1_mean']:.3f}  Brier {v['brier_mean']:.4f}  ECE {v['ece_mean']:.3f}")


## Read it
- **physics** is the calibrated fast-marching prior; **corrector** is the U-Net that also
  sees the suppression/barrier layers. If corrector > physics on AP with lower ECE, the
  learned residual is real; if not, the physics prior alone is the honest answer.
- Everything is leave-fire-out and scored on new-burn only, so numbers are comparable to
  the WildfireSpreadTS literature. Report AP + calibration (Brier/ECE), not just F1.
